# Extension 3 — LLM→Segmentation Feedback Loop

**Thesis Section: Chapter 6.3 — Classification-guided localisation**

The feedback loop closes the gap between coarse classification (Approach 3)
and fine-grained localisation (MLP segmentation):

```
Signal
  └─► Approach 3 (ChatTS-14B)  →  Category  (e.g. A = Drift)
         └─► query_selector     →  Query     (e.g. 'Find gradual baseline drift')
               └─► MLP segmentation          →  Probability map + binary mask
```

**Without feedback**: segmentation always uses a fixed query per experiment.
**With feedback**: the query is automatically selected based on what the
classifier found — so a frozen sensor gets the frozen query, a drifting
sensor gets the drift query, etc.

**Hypothesis**: Category-aware queries improve segmentation F1 vs a single
generic query, especially for multi-class datasets.

**Requires**: `segmentation_mlp_final.pt` on VSC scratch.

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.timeseer_client import fetch_series_api
from src.data.ground_truth import GROUND_TRUTH
from src.models.chatts_loader import load_model
from src.models.mlp import load_mlp
from src.prescreener.analyze import hybrid_analyze
from src.inference.chatts_infer import ask_chatts_chunk
from src.inference.embeddings import encode_text_query
from src.inference.chronos_server import start_server, get_chronos_embedding_cached, shutdown_server
from src.parsing.extract import extract_category
from src.preprocessing.chunking import downsample
from src.segmentation.query_selector import feedback_segment, multiscale_feedback_segment, CATEGORY_QUERIES
from src.segmentation.pipeline import segment_signal
from src.visualization.segmentation_plots import plot_segmentation

VSC_SCRATCH = os.environ.get('VSC_SCRATCH', '/scratch/leuven/375/vsc37531')
MLP_PATH    = os.path.join(VSC_SCRATCH, 'ChatTS', 'timeseer_data', 'segmentation_mlp_final.pt')
THRESHOLD   = 0.65

print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
model, tokenizer, processor = load_model('ChatTS-14B')
def _encode(q): return encode_text_query(q, model=model, tokenizer=tokenizer)

start_server()
mlp = load_mlp(MLP_PATH)

print('\nCategory → Query mapping:')
for cat, q in CATEGORY_QUERIES.items():
    print(f'  {cat}: {q}')

In [ ]:
# Labelled signals with known ground-truth anomaly ranges
TEST_SIGNALS = [
    ('R1-AT-101-PH',   'Reactor 1', 'A', 0,    8640),   # Drift
    ('R1-AT-103-DO',   'Reactor 1', 'C', 2131, 2259),   # Frozen
    ('R1-AT-102-COND', 'Reactor 1', 'B', 1460, 1475),   # Spikes
]

results = []

for tag, area, gt_cat, gt_start, gt_end in TEST_SIGNALS:
    print(f'\n── {tag} (GT={gt_cat}) ──')
    vals, idx = fetch_series_api(tag, area)
    vals_ds   = downsample(vals, target=1024)
    n = len(vals_ds)

    # Scale ground-truth to downsampled space
    true_s = int(gt_start * n / len(vals))
    true_e = int(gt_end   * n / len(vals))
    true_mask = np.zeros(n)
    true_mask[true_s:true_e] = 1.0

    # ── Step 1: Classify with Approach 3 ──────────────────────────
    chunk, question, tname, detected, chunk_desc = hybrid_analyze(vals, idx, tag)
    torch.cuda.empty_cache()
    answer = ask_chatts_chunk(chunk, question,
                              model=model, tokenizer=tokenizer, processor=processor)
    pred_cat, pred_lbl = extract_category(answer, detected)
    print(f'  Classification: {pred_cat}) {pred_lbl}  (GT: {gt_cat})')

    # ── Step 2: Category-guided segmentation (feedback) ────────────
    prob_fb, mask_fb, query_used = feedback_segment(
        vals_ds, pred_cat, mlp, _encode, get_chronos_embedding_cached,
        window_size=32, threshold=THRESHOLD,
    )
    print(f'  Query selected: "{query_used}"')

    # ── Step 3: Baseline — generic query segmentation ──────────────
    GENERIC_QUERY = 'Find periods with abnormal dynamics'
    prob_gen, mask_gen = segment_signal(
        vals_ds, GENERIC_QUERY, mlp, _encode, get_chronos_embedding_cached,
        window_size=32, threshold=THRESHOLD,
    )

    # ── Step 4: Oracle — correct-category query segmentation ───────
    from src.segmentation.query_selector import CATEGORY_QUERIES
    oracle_query = CATEGORY_QUERIES.get(gt_cat, GENERIC_QUERY)
    prob_oracle, mask_oracle = segment_signal(
        vals_ds, oracle_query, mlp, _encode, get_chronos_embedding_cached,
        window_size=32, threshold=THRESHOLD,
    )

    # F1 for each condition
    def f1_score(prob, mask):
        if prob is None:
            return 0.0
        binary = (prob > THRESHOLD).astype(float)
        tp = ((binary == 1) & (mask == 1)).sum()
        fp = ((binary == 1) & (mask == 0)).sum()
        fn = ((binary == 0) & (mask == 1)).sum()
        pr = tp / (tp + fp + 1e-8)
        rc = tp / (tp + fn + 1e-8)
        return float(2 * pr * rc / (pr + rc + 1e-8))

    f1_generic  = f1_score(prob_gen,    true_mask)
    f1_feedback = f1_score(prob_fb,     true_mask)
    f1_oracle   = f1_score(prob_oracle, true_mask)

    print(f'  F1 generic   : {f1_generic:.3f}')
    print(f'  F1 feedback  : {f1_feedback:.3f}  (pred_cat={pred_cat})')
    print(f'  F1 oracle    : {f1_oracle:.3f}  (gt_cat={gt_cat})')

    results.append({
        'tag': tag, 'gt_cat': gt_cat, 'pred_cat': pred_cat,
        'f1_generic': round(f1_generic, 3),
        'f1_feedback': round(f1_feedback, 3),
        'f1_oracle':  round(f1_oracle, 3),
        'query_used': query_used,
    })

In [ ]:
# Summary table
print('\n' + '='*75)
print('FEEDBACK LOOP RESULTS — Segmentation F1')
print('='*75)
print(f'{"Signal":<22} {"GT":<4} {"Pred":<5} {"Generic":<10} {"Feedback":<10} {"Oracle":<8}')
print('-'*75)
for r in results:
    match = '✓' if r['pred_cat'] == r['gt_cat'] else '✗'
    print(f'{r["tag"]:<22} {r["gt_cat"]:<4} '
          f'{r["pred_cat"]}{match:<4} '
          f'{r["f1_generic"]:.3f}      '
          f'{r["f1_feedback"]:.3f}      '
          f'{r["f1_oracle"]:.3f}')
print('-'*75)

import numpy as np
avg_gen  = np.mean([r['f1_generic']  for r in results])
avg_fb   = np.mean([r['f1_feedback'] for r in results])
avg_ora  = np.mean([r['f1_oracle']   for r in results])
print(f'{"Average":<22} {"":<4} {"":<5} {avg_gen:.3f}      {avg_fb:.3f}      {avg_ora:.3f}')
print('\nFeedback delta vs generic: ', round(avg_fb - avg_gen, 3))
print('Oracle delta vs generic:   ', round(avg_ora - avg_gen, 3))

shutdown_server()

In [ ]:
# Bar chart: Generic vs Feedback vs Oracle F1
import matplotlib.pyplot as plt
import numpy as np

tags  = [r['tag'].replace('R1-AT-', '') for r in results]
x     = np.arange(len(tags))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width, [r['f1_generic']  for r in results], width, label='Generic query',  color='#4C78A8')
ax.bar(x,         [r['f1_feedback'] for r in results], width, label='Feedback query', color='#F58518')
ax.bar(x + width, [r['f1_oracle']   for r in results], width, label='Oracle query',   color='#54A24B', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(tags)
ax.set_ylabel('Segmentation F1')
ax.set_ylim(0, 1.05)
ax.set_title('Feedback Loop: Classification-Guided Segmentation')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../../data/figures/ext3_feedback_loop_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure.')